## Importacion de Libreria

In [1]:
import json
import requests
import pyodbc
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
import os

In [30]:
#Obtencion de servidor Sql y API de https://api.football-data.org

load_dotenv()  # ← carga el .env

API_KEY = os.getenv("FOOTBALL_API_KEY")  # ← define la variable
DB_SERVER = os.getenv("DB_SERVER")
DB_NAME = os.getenv("DB_NAME")

In [4]:
# Conectar la info del API al SQL y guardar los datos en la base de datos para su posterior análisis.

conn = pyodbc.connect(
    f"DRIVER={{SQL Server}};"
    f"SERVER={DB_SERVER};"
    f"DATABASE={DB_NAME};"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()

## Obtener ID de la liga española

In [4]:
### Extracting data from API
URL = 'https://api.football-data.org/v4/competitions'
credenciales = {'X-Auth-Token': API_KEY}

response = requests.get(URL,headers=credenciales)
data = response.json()


In [5]:
print(json.dumps(data['competitions'][0], indent=2))

{
  "id": 2013,
  "area": {
    "id": 2032,
    "name": "Brazil",
    "code": "BRA",
    "flag": "https://crests.football-data.org/764.svg"
  },
  "name": "Campeonato Brasileiro S\u00e9rie A",
  "code": "BSA",
  "type": "LEAGUE",
  "emblem": "https://crests.football-data.org/bsa.png",
  "plan": "TIER_ONE",
  "currentSeason": {
    "id": 2474,
    "startDate": "2026-01-28",
    "endDate": "2026-12-02",
    "currentMatchday": 6,
    "winner": null
  },
  "numberOfAvailableSeasons": 10,
  "lastUpdated": "2024-09-13T16:55:53Z"
}


In [6]:
for competicion in data['competitions']:
    print(f"ID: {competicion['id']}, Name: {competicion['name']}") # El ID de la Liga Española es 2014

ID: 2013, Name: Campeonato Brasileiro Série A
ID: 2016, Name: Championship
ID: 2021, Name: Premier League
ID: 2001, Name: UEFA Champions League
ID: 2018, Name: European Championship
ID: 2015, Name: Ligue 1
ID: 2002, Name: Bundesliga
ID: 2019, Name: Serie A
ID: 2003, Name: Eredivisie
ID: 2017, Name: Primeira Liga
ID: 2152, Name: Copa Libertadores
ID: 2014, Name: Primera Division
ID: 2000, Name: FIFA World Cup


El ID que utilizaremos es el 2014 ya que estamos interesados en la liga española

## Insertar Datos de Competicion
* Informacion de la Liga Española

In [7]:
URL = 'https://api.football-data.org/v4/competitions/2014'
response = requests.get(URL, headers=credenciales)
comp = response.json()
print(json.dumps(comp, indent=2))

{
  "area": {
    "id": 2224,
    "name": "Spain",
    "code": "ESP",
    "flag": "https://crests.football-data.org/760.svg"
  },
  "id": 2014,
  "name": "Primera Division",
  "code": "PD",
  "type": "LEAGUE",
  "emblem": "https://crests.football-data.org/laliga.png",
  "currentSeason": {
    "id": 2429,
    "startDate": "2025-08-17",
    "endDate": "2026-05-24",
    "currentMatchday": 28,
    "winner": null
  },
  "seasons": [
    {
      "id": 2429,
      "startDate": "2025-08-17",
      "endDate": "2026-05-24",
      "currentMatchday": 28,
      "winner": null
    },
    {
      "id": 2292,
      "startDate": "2024-08-18",
      "endDate": "2025-05-25",
      "currentMatchday": 38,
      "winner": null
    },
    {
      "id": 1577,
      "startDate": "2023-08-13",
      "endDate": "2024-05-26",
      "currentMatchday": 38,
      "winner": null
    },
    {
      "id": 1504,
      "startDate": "2022-08-14",
      "endDate": "2023-06-04",
      "currentMatchday": 38,
      "winner": 

In [8]:
for key, value in comp.items():
    print(f"{key}: {value}")

area: {'id': 2224, 'name': 'Spain', 'code': 'ESP', 'flag': 'https://crests.football-data.org/760.svg'}
id: 2014
name: Primera Division
code: PD
type: LEAGUE
emblem: https://crests.football-data.org/laliga.png
currentSeason: {'id': 2429, 'startDate': '2025-08-17', 'endDate': '2026-05-24', 'currentMatchday': 28, 'winner': None}
seasons: [{'id': 2429, 'startDate': '2025-08-17', 'endDate': '2026-05-24', 'currentMatchday': 28, 'winner': None}, {'id': 2292, 'startDate': '2024-08-18', 'endDate': '2025-05-25', 'currentMatchday': 38, 'winner': None}, {'id': 1577, 'startDate': '2023-08-13', 'endDate': '2024-05-26', 'currentMatchday': 38, 'winner': None}, {'id': 1504, 'startDate': '2022-08-14', 'endDate': '2023-06-04', 'currentMatchday': 38, 'winner': None}, {'id': 380, 'startDate': '2021-08-13', 'endDate': '2022-05-22', 'currentMatchday': 38, 'winner': None}, {'id': 635, 'startDate': '2020-09-12', 'endDate': '2021-05-23', 'currentMatchday': 38, 'winner': {'id': 78, 'name': 'Club Atlético de Madr

In [ ]:
cursor.execute("""
    INSERT INTO dim_competicion  (id_competicion, nombre, codigo , pais)
    VALUES (?, ?, ?, ?)
""", 
    comp['id'], 
    comp['name'], 
    comp['code'], 
    comp['area']['name']
    )
conn.commit()
print("Informacion de la competicion subida al Sql Server Correctamente")

In [6]:
URL = 'https://api.football-data.org/v4/competitions/2014/matches' # se considera el ID en la URL para obtener los partidos de la Liga Española

credenciales = {'X-Auth-Token': API_KEY}

response = requests.get(URL,headers=credenciales)
data = response.json()

In [7]:
data

{'filters': {'season': '2025'},
 'resultSet': {'count': 380,
  'first': '2025-08-15',
  'last': '2026-05-24',
  'played': 275},
 'competition': {'id': 2014,
  'name': 'Primera Division',
  'code': 'PD',
  'type': 'LEAGUE',
  'emblem': 'https://crests.football-data.org/laliga.png'},
 'matches': [{'area': {'id': 2224,
    'name': 'Spain',
    'code': 'ESP',
    'flag': 'https://crests.football-data.org/760.svg'},
   'competition': {'id': 2014,
    'name': 'Primera Division',
    'code': 'PD',
    'type': 'LEAGUE',
    'emblem': 'https://crests.football-data.org/laliga.png'},
   'season': {'id': 2429,
    'startDate': '2025-08-17',
    'endDate': '2026-05-24',
    'currentMatchday': 28,
    'winner': None},
   'id': 544214,
   'utcDate': '2025-08-15T17:00:00Z',
   'status': 'FINISHED',
   'matchday': 1,
   'stage': 'REGULAR_SEASON',
   'group': None,
   'lastUpdated': '2026-03-14T00:21:02Z',
   'homeTeam': {'id': 298,
    'name': 'Girona FC',
    'shortName': 'Girona',
    'tla': 'GIR',
 

In [8]:
print(json.dumps(data['matches'][0], indent=2))

{
  "area": {
    "id": 2224,
    "name": "Spain",
    "code": "ESP",
    "flag": "https://crests.football-data.org/760.svg"
  },
  "competition": {
    "id": 2014,
    "name": "Primera Division",
    "code": "PD",
    "type": "LEAGUE",
    "emblem": "https://crests.football-data.org/laliga.png"
  },
  "season": {
    "id": 2429,
    "startDate": "2025-08-17",
    "endDate": "2026-05-24",
    "currentMatchday": 28,
    "winner": null
  },
  "id": 544214,
  "utcDate": "2025-08-15T17:00:00Z",
  "status": "FINISHED",
  "matchday": 1,
  "stage": "REGULAR_SEASON",
  "group": null,
  "lastUpdated": "2026-03-14T00:21:02Z",
  "homeTeam": {
    "id": 298,
    "name": "Girona FC",
    "shortName": "Girona",
    "tla": "GIR",
    "crest": "https://crests.football-data.org/298.png"
  },
  "awayTeam": {
    "id": 87,
    "name": "Rayo Vallecano de Madrid",
    "shortName": "Rayo Vallecano",
    "tla": "RAY",
    "crest": "https://crests.football-data.org/87.png"
  },
  "score": {
    "winner": "AWA

In [9]:
for partido in data['matches'][:5]:
    print(f'Jornada: {partido["matchday"]} | {partido["homeTeam"]["name"]} vs {partido["awayTeam"]["name"]} | -> {partido["score"]["fullTime"]["home"]} - {partido["score"]["fullTime"]["away"]}')


Jornada: 1 | Girona FC vs Rayo Vallecano de Madrid | -> 1 - 3
Jornada: 1 | Villarreal CF vs Real Oviedo | -> 2 - 0
Jornada: 1 | RCD Mallorca vs FC Barcelona | -> 0 - 3
Jornada: 1 | Deportivo Alavés vs Levante UD | -> 2 - 1
Jornada: 1 | Valencia CF vs Real Sociedad de Fútbol | -> 1 - 1


In [13]:
for partido in data['matches']:

    id_partido  = partido['id']
    fecha       = partido['utcDate']
    local       = partido['homeTeam']['name']
    visitante   = partido['awayTeam']['name']
    goles_local = partido['score']['fullTime']['home']
    goles_visita = partido['score']['fullTime']['away']

    cursor.execute("""
        INSERT INTO fact_partido (id_partido, fecha, local, visitante, goles_local, goles_visitante)
        VALUES (?, ?, ?, ?, ?, ?)
    """, id_partido, fecha, local, visitante, goles_local, goles_visita)

conn.commit()


ProgrammingError: ('42S22', "[42S22] [Microsoft][ODBC SQL Server Driver][SQL Server]Invalid column name 'local'. (207) (SQLExecDirectW); [42S22] [Microsoft][ODBC SQL Server Driver][SQL Server]Invalid column name 'visitante'. (207); [42S22] [Microsoft][ODBC SQL Server Driver][SQL Server]Statement(s) could not be prepared. (8180)")

## Informacion de los equipos Españoles

In [12]:
URL = 'https://api.football-data.org/v4/competitions/2014/teams'
token = API_KEY
credenciales = {'X-Auth-Token': token}

response = requests.get(URL,headers=credenciales)
data = response.json()

In [13]:
for key, value in data.items():
    print(f"{key}: {value}")

count: 20
filters: {'season': '2025'}
competition: {'id': 2014, 'name': 'Primera Division', 'code': 'PD', 'type': 'LEAGUE', 'emblem': 'https://crests.football-data.org/laliga.png'}
season: {'id': 2429, 'startDate': '2025-08-17', 'endDate': '2026-05-24', 'currentMatchday': 28, 'winner': None}
teams: [{'area': {'id': 2224, 'name': 'Spain', 'code': 'ESP', 'flag': 'https://crests.football-data.org/760.svg'}, 'id': 77, 'name': 'Athletic Club', 'shortName': 'Athletic', 'tla': 'ATH', 'crest': 'https://crests.football-data.org/77.png', 'address': 'Ibaigane, Alameda Mazarredo, 23 Bilbao 48009', 'website': 'http://www.athletic-club.eus', 'founded': 1898, 'clubColors': 'Red / White / Black', 'venue': 'San Mamés', 'runningCompetitions': [{'id': 2014, 'name': 'Primera Division', 'code': 'PD', 'type': 'LEAGUE', 'emblem': 'https://crests.football-data.org/laliga.png'}, {'id': 2001, 'name': 'UEFA Champions League', 'code': 'CL', 'type': 'CUP', 'emblem': 'https://crests.football-data.org/CL.png'}], 'co

In [14]:
print(json.dumps(data['teams'][0], indent=2))

{
  "area": {
    "id": 2224,
    "name": "Spain",
    "code": "ESP",
    "flag": "https://crests.football-data.org/760.svg"
  },
  "id": 77,
  "name": "Athletic Club",
  "shortName": "Athletic",
  "tla": "ATH",
  "crest": "https://crests.football-data.org/77.png",
  "address": "Ibaigane, Alameda Mazarredo, 23 Bilbao 48009",
  "website": "http://www.athletic-club.eus",
  "founded": 1898,
  "clubColors": "Red / White / Black",
  "venue": "San Mam\u00e9s",
  "runningCompetitions": [
    {
      "id": 2014,
      "name": "Primera Division",
      "code": "PD",
      "type": "LEAGUE",
      "emblem": "https://crests.football-data.org/laliga.png"
    },
    {
      "id": 2001,
      "name": "UEFA Champions League",
      "code": "CL",
      "type": "CUP",
      "emblem": "https://crests.football-data.org/CL.png"
    }
  ],
  "coach": {
    "id": 15873,
    "firstName": "",
    "lastName": "Ernesto Valverde",
    "name": "Ernesto Valverde",
    "dateOfBirth": "1964-02-09",
    "nationality":

In [16]:
for equipo in data['teams']:
    id_equipo = equipo['id']
    nombre = equipo['name']
    nombre_corto = equipo['shortName']
    estadio = equipo['venue']
    cursor.execute("""
        INSERT INTO dim_equipo (id_equipo, nombre, nombre_corto, estadio)
        VALUES (?, ?, ?, ?)
    """, id_equipo, nombre, nombre_corto, estadio)
    conn.commit()

In [25]:
df_equipos = pd.read_sql_query("SELECT * FROM dim_equipo", conn)
df_equipos

C:\Users\brger\AppData\Local\Temp\ipykernel_15200\2982039710.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_equipos = pd.read_sql_query("SELECT * FROM dim_equipo", conn)


,id_equipo,nombre,nombre_corto,estadio
0,77,Athletic Club,Athletic,San Mamés
1,78,Club Atlético de Madrid,Atleti,Estadio Wanda Metropolitano
2,79,CA Osasuna,Osasuna,Estadio El Sadar
3,80,RCD Espanyol de Barcelona,Espanyol,RCDE Stadium
4,81,FC Barcelona,Barça,Camp Nou
5,82,Getafe CF,Getafe,Coliseum Alfonso Pérez
6,86,Real Madrid CF,Real Madrid,Estadio Santiago Bernabéu
7,87,Rayo Vallecano de Madrid,Rayo Vallecano,Estadio del Rayo Vallecano
8,88,Levante UD,Levante,Estadio Ciudad de Valencia
9,89,RCD Mallorca,Mallorca,Iberostar Estadi


## Informacion de los partidos de la Liga

In [31]:
URL = 'https://api.football-data.org/v4/competitions/2014/matches'
credenciales = {'X-Auth-Token': API_KEY}
response = requests.get(URL,headers=credenciales)
partidos = response.json()

In [27]:
json.dumps(partidos['matches'][0], indent=2)

'{\n  "area": {\n    "id": 2224,\n    "name": "Spain",\n    "code": "ESP",\n    "flag": "https://crests.football-data.org/760.svg"\n  },\n  "competition": {\n    "id": 2014,\n    "name": "Primera Division",\n    "code": "PD",\n    "type": "LEAGUE",\n    "emblem": "https://crests.football-data.org/laliga.png"\n  },\n  "season": {\n    "id": 2429,\n    "startDate": "2025-08-17",\n    "endDate": "2026-05-24",\n    "currentMatchday": 28,\n    "winner": null\n  },\n  "id": 544214,\n  "utcDate": "2025-08-15T17:00:00Z",\n  "status": "FINISHED",\n  "matchday": 1,\n  "stage": "REGULAR_SEASON",\n  "group": null,\n  "lastUpdated": "2026-03-14T00:21:02Z",\n  "homeTeam": {\n    "id": 298,\n    "name": "Girona FC",\n    "shortName": "Girona",\n    "tla": "GIR",\n    "crest": "https://crests.football-data.org/298.png"\n  },\n  "awayTeam": {\n    "id": 87,\n    "name": "Rayo Vallecano de Madrid",\n    "shortName": "Rayo Vallecano",\n    "tla": "RAY",\n    "crest": "https://crests.football-data.org/87.

In [33]:
cursor = conn.cursor()

mapa = {'HOME_TEAM': 'H', 'AWAY_TEAM': 'A', 'DRAW': 'D'}

for partido in partidos['matches']:
    if partido['status'] != 'FINISHED':
        continue

    temporada       = '2025/26'
    fecha           = partido['utcDate'][:10]
    jornada         = partido['matchday']
    id_local        = partido['homeTeam']['id']
    id_visitante    = partido['awayTeam']['id']
    goles_local     = partido['score']['fullTime']['home']
    goles_visitante = partido['score']['fullTime']['away']
    goles_ht_local  = partido['score']['halfTime']['home']
    goles_ht_visit  = partido['score']['halfTime']['away']
    resultado       = mapa[partido['score']['winner']]

    cursor.execute("""
        INSERT INTO fact_partido (
            temporada, fecha, jornada,
            id_equipo_local, id_equipo_visitante,
            goles_local, goles_visitante, resultado,
            goles_ht_local, goles_ht_visitante,
            fuente
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, temporada, fecha, jornada,
         id_local, id_visitante,
         goles_local, goles_visitante, resultado,
         goles_ht_local, goles_ht_visit,
         'API')

conn.commit()
print("✅ fact_partido cargada")

✅ fact_partido cargada


## Informacion por Jornada

In [ ]:
URL = 'https://api.football-data.org//v4/competitions/2014/standings?matchday=1'
credenciales = {'X-Auth-Token': API_KEY}
response = requests.get(URL,headers=credenciales)
standings = response.json()

In [39]:
for equipo in standings:
    print(equipo)
    

{'position': 1, 'team': {'id': 81, 'name': 'FC Barcelona', 'shortName': 'Barça', 'tla': 'FCB', 'crest': 'https://crests.football-data.org/81.png'}, 'playedGames': 27, 'form': None, 'won': 22, 'draw': 1, 'lost': 4, 'points': 67, 'goalsFor': 72, 'goalsAgainst': 26, 'goalDifference': 46}
{'position': 2, 'team': {'id': 86, 'name': 'Real Madrid CF', 'shortName': 'Real Madrid', 'tla': 'RMA', 'crest': 'https://crests.football-data.org/86.png'}, 'playedGames': 28, 'form': None, 'won': 21, 'draw': 3, 'lost': 4, 'points': 66, 'goalsFor': 60, 'goalsAgainst': 24, 'goalDifference': 36}
{'position': 3, 'team': {'id': 78, 'name': 'Club Atlético de Madrid', 'shortName': 'Atleti', 'tla': 'ATL', 'crest': 'https://crests.football-data.org/78.png'}, 'playedGames': 28, 'form': None, 'won': 17, 'draw': 6, 'lost': 5, 'points': 57, 'goalsFor': 47, 'goalsAgainst': 25, 'goalDifference': 22}
{'position': 4, 'team': {'id': 94, 'name': 'Villarreal CF', 'shortName': 'Villarreal', 'tla': 'VIL', 'crest': 'https://cre

 Los datos sacados del API te dan los el detalle del global de la campaña no de la jornada
 Por ende se sacara el detalle desde la tabla fact_partido y se calculara el standing jornada a jornada para luego comparar con el standing que da el API y ver si coincide o no. Esto se hara en la fase de Transformacion.

## Informacion de los partidos

In [45]:
df = pd.read_csv(r'C:\Users\brger\OneDrive\Desktop\Project\Football_Prediction\Data\SP1.csv',sep=',')
df.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,SP1,15/08/2025,18:00,Girona,Vallecano,1,3,A,0,3,...,2.1,1.78,1.97,1.95,2.10,1.80,1.95,1.75,2.11,1.89
1,SP1,15/08/2025,20:30,Villarreal,Oviedo,2,0,H,2,0,...,1.9,1.95,1.93,2.00,1.94,1.96,1.82,1.87,1.96,2.02
2,SP1,16/08/2025,18:30,Mallorca,Barcelona,0,3,A,0,2,...,2.0,1.85,1.88,2.05,2.00,1.93,1.90,1.86,2.04,1.94
3,SP1,16/08/2025,20:30,Alaves,Levante,2,1,H,1,0,...,1.8,2.05,1.86,2.07,1.80,2.07,1.70,2.00,1.81,2.20
4,SP1,16/08/2025,20:30,Valencia,Sociedad,1,1,D,0,0,...,2.1,1.78,2.13,1.81,2.10,1.78,1.96,1.74,2.18,1.84


In [46]:
print(sorted(df['HomeTeam'].unique()))

['Alaves', 'Ath Bilbao', 'Ath Madrid', 'Barcelona', 'Betis', 'Celta', 'Elche', 'Espanol', 'Getafe', 'Girona', 'Levante', 'Mallorca', 'Osasuna', 'Oviedo', 'Real Madrid', 'Sevilla', 'Sociedad', 'Valencia', 'Vallecano', 'Villarreal']


In [47]:
df_equipos = pd.read_sql_query("SELECT id_equipo, nombre, nombre_corto FROM dim_equipo", conn)
print(df_equipos.to_string())

    id_equipo                     nombre    nombre_corto
0          77              Athletic Club        Athletic
1          78    Club Atlético de Madrid          Atleti
2          79                 CA Osasuna         Osasuna
3          80  RCD Espanyol de Barcelona        Espanyol
4          81               FC Barcelona           Barça
5          82                  Getafe CF          Getafe
6          86             Real Madrid CF     Real Madrid
7          87   Rayo Vallecano de Madrid  Rayo Vallecano
8          88                 Levante UD         Levante
9          89               RCD Mallorca        Mallorca
10         90        Real Betis Balompié      Real Betis
11         92    Real Sociedad de Fútbol   Real Sociedad
12         94              Villarreal CF      Villarreal
13         95                Valencia CF        Valencia
14        263           Deportivo Alavés          Alavés
15        285                   Elche CF           Elche
16        298                  

C:\Users\brger\AppData\Local\Temp\ipykernel_15200\2390548524.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_equipos = pd.read_sql_query("SELECT id_equipo, nombre, nombre_corto FROM dim_equipo", conn)
